# ViT Position Encoding Analysis: SSDC, RPI, and Layer Windowed Ablation

This notebook reproduces the spatial structure analysis for two pretrained Vision Transformers, then runs a new mechanistic ablation experiment.

**Models**
- APE: `google/vit-base-patch16-224` (learned absolute position embeddings), via transformers.
- RoPE: `vit_base_patch16_rope_224.naver_in1k` (rotary position embeddings), via timm.

**Data**: ImageNet-1k validation, streamed from the Hugging Face Hub (gated, needs a token).

**What runs here**
1. SSDC across depth, clean and under Random Permutation at Inference (RPI), for both models.
2. Robustness (fragility) under ImageNet-C Gaussian blur.
3. The new experiment: zero ablating MLP or attention sublayers in early, middle, or late layer windows for the APE model. This asks whether the early SSDC peak is tied to the early MLPs specifically or just to the first MLPs the tokens meet, and whether removing attention only in later layers still flattens the later layer SSDC decay.
4. Extension: effective rank across depth and its collapse under MLP ablation (Dong et al., 2021).

Guidance paper: Mannes, *Positional Encodings Anchor Spatial Structure in Vision Transformers* (arXiv 2606.00124). The experiments here are our own and use pretrained ViT-Base models, not the from scratch ViT-S setup in that paper.

Use a GPU runtime (Runtime > Change runtime type > GPU).

## 0. Setup

In [ ]:
# Colab already ships torch, numpy, scipy, matplotlib, Pillow.
!pip -q install timm transformers datasets scikit-image huggingface_hub

In [ ]:
import os, sys

REPO = 'vit-sae-analysis'
if not os.path.exists(REPO):
    !git clone https://github.com/aravinds-kannappan/vit-sae-analysis.git

SRC = os.path.abspath(os.path.join(REPO, 'project_code', 'src'))
if SRC not in sys.path:
    sys.path.insert(0, SRC)
print('src on path:', SRC)

# Optional: mount Drive to persist figures and results.
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# ImageNet-1k is gated. Your token must be able to READ gated repos:
#   1. Accept the terms once at https://huggingface.co/datasets/ILSVRC/imagenet-1k
#   2. Use EITHER a classic 'Read' token, OR a fine-grained token with
#      'Read access to contents of all public gated repos you can access'
#      enabled at https://huggingface.co/settings/tokens
# If the token still cannot read it, the dataset cell auto-falls back to an
# ungated mirror (benjamin-paine/imagenet-1k-256x256) so the notebook still runs.
import os, getpass
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass.getpass('Hugging Face token: ')
from huggingface_hub import login
login(os.environ['HF_TOKEN'], add_to_git_credential=False)

In [ ]:
import torch
import matplotlib.pyplot as plt
from experiments import common, reproduce_ssdc, reproduce_robustness, ablation_layerwise, effective_rank_probe

print('device:', 'cuda' if torch.cuda.is_available() else 'cpu (slow, switch to a GPU runtime)')

# Sample sizes. Raise for smoother curves, lower for a quick pass.
NUM_IMAGES = 1000
BATCH_SIZE = 128
ABLATION_IMAGES = 512   # the ablation sweep runs many conditions, keep it modest

# Stream ImageNet-1k validation once and reuse it (re-iterating restarts the stream).
# ILSVRC/imagenet-1k is gated. If your account has not accepted its terms you get a
# 403, and this call auto-falls back to the ungated mirror
# benjamin-paine/imagenet-1k-256x256 (same schema and 0-999 labels). To use the
# official split, click 'Agree and access repository' at
# https://huggingface.co/datasets/ILSVRC/imagenet-1k then rerun this cell.
dataset = common.load_imagenet(split='validation', streaming=True)

## 1. SSDC across depth, clean and under RPI

APE should show SSDC under RPI peaking early (blocks 2 to 3) then decaying with depth. RoPE should accumulate more gradually and peak later. Dashed lines are the reference curves stored in the repo.

In [ ]:
ssdc_results = {}
for kind in ['ape', 'rope']:
    print(f'=== {kind.upper()} ===')
    ssdc_results[kind] = reproduce_ssdc.run_model(kind, dataset, number_images=NUM_IMAGES, batch_size=BATCH_SIZE)
    print('clean:', [round(x, 3) for x in ssdc_results[kind]['clean']])
    print('rpi:  ', [round(x, 3) for x in ssdc_results[kind]['rpi']])
    reproduce_ssdc.plot_model(kind, ssdc_results[kind])
plt.show()

## 2. Robustness under Gaussian blur

Fragility = 1 - shifted / baseline accuracy under ImageNet-C Gaussian blur (severity 5). Lower is more robust. Reference from the original run: APE 0.326, RoPE 0.299. In both the reference and a fresh run (APE 0.385, RoPE 0.281), RoPE is the more robust model, and it is also the one that keeps its index anchored frame deepest (section 1), which lines up with the stable reference frame story.

In [ ]:
rob = {}
for kind in ['ape', 'rope']:
    rob[kind] = reproduce_robustness.run_model(kind, dataset, corruption='Gaussian Blur', severity=5,
                                               number_images=NUM_IMAGES, batch_size=BATCH_SIZE)

print('\nmodel   baseline  shifted  fragility')
for kind in ['ape', 'rope']:
    r = rob[kind]
    print(f"{kind:5s}   {r['baseline_accuracy']:.3f}    {r['shifted_accuracy']:.3f}    {r['fragility']:.3f}")

## 3. New experiment: layer windowed MLP and attention ablation (APE)

Windows: early = [0,1,2,3], mid = [4,5,6,7], late = [8,9,10,11]. Primary metric is SSDC under RPI, since both the early peak and the later decay live in that curve for APE. This runs about twelve conditions, so give it a few minutes on a GPU. Three figures are produced: MLP ablation by window, keep MLPs in one window only, and attention ablation by window.

In [ ]:
abl = ablation_layerwise.run_all(dataset, model_kind='ape',
                                 number_images=ABLATION_IMAGES, batch_size=BATCH_SIZE,
                                 do_rank=False, do_plot=True)
plt.show()

### Reading the ablation result: what the run actually showed

The run reversed the two starting claims. Full write up in `docs/DISCUSSION.md`.

- **The early peak is attention, not MLPs.** Every `mlp_*` condition still peaks near 0.78 at block 3, including `mlp_zero_all`. The peak only moves when attention is ablated: `attn_zero_all` drops it to 0.62 at block 6, and `attn_zero_early` turns the immediate jump negative. So the early SSDC under RPI peak is built by early attention reading the added position vectors, not by MLPs. That answers the motivating question directly: the peak is neither unique to the early MLPs nor a first MLPs encountered effect, it is not an MLP effect at all, since it survives `mlp_zero_all`.
- **The later decay is the middle MLPs.** `mlp_zero_mid` cuts the decay from 0.582 to 0.301 (final block 0.204 to 0.476), and `mlp_zero_all` nearly erases it (final 0.677), while `mlp_zero_late` does nothing. The keep only probe agrees: keeping MLPs alive only in the mid window reproduces the decay, keeping them only early or only late does not. Middle block MLPs overwrite the index anchored structure with content and task features.
- **Late attention only partly softens the decay.** `attn_zero_late` keeps the early peak and lifts the final block from 0.204 to 0.337, so it reduces but does not flatten the decay. `attn_zero_all` shrinks the decay mostly by pulling the peak down, a different mechanism from the MLP case.

Bottom line: attention builds the early recovery and the middle MLPs drive its decay. The summary table (peak, peak_L, delta, decay, final, auc) backs each read.

## 4. Extension: effective rank and rank collapse (Dong et al., 2021)

Attention without MLPs drives token representations toward rank one with depth. This run reproduces that: `mlp_zero_all` collapses effective rank to about 43 at the last block, while `attn_zero_all` keeps it high (about 136). The twist is the cross reference with the ablation above. `mlp_zero_all` is the same condition that keeps SSDC under RPI highest, so rank collapse and loss of index anchored structure are dissociated. Low rank representations can still be strongly organized by token index, which means SSDC under RPI is not just a proxy for representational rank.

In [ ]:
rank_curves = effective_rank_probe.run('ape', dataset, number_images=ABLATION_IMAGES,
                                       batch_size=BATCH_SIZE, rpi=True)
common.plot_curves(rank_curves, title='APE effective rank across depth (under RPI)', ylabel='effective rank')
plt.show()

## 5. Discussion

Full write up: `docs/DISCUSSION.md`. Short version:

- **APE vs RoPE.** APE injects index anchored structure up front (SSDC under RPI peaks at block 3 to 4, about 0.82) then loses it (final 0.20). RoPE accumulates it gradually (peak block 5, about 0.64) and holds it (final 0.47), because it re applies position inside every attention layer.
- **Robustness tracks persistence.** RoPE is both less fragile under Gaussian blur (0.281 vs 0.385) and the model that keeps its index frame deeper. Consistent with the paper's stable reference frame thesis, though only two models here.
- **Mechanism (APE).** Early attention builds the recovery, the middle MLPs drive the decay, and whether the peak or the tail moves is what separates the two ablations. This overturns the starting assumption that MLPs carry the recovery.
- **Rank is a separate axis.** Removing MLPs collapses effective rank but keeps index anchored SSDC high, so the two probes measure different things.

Caveats: single run per condition, pretrained ViT-Base rather than the paper's from scratch ViT-S, zero ablation, one corruption. Next: sweep the mid window boundary block by block, repeat on RoPE, add a content control, and tie the middle MLP features to the SAE analysis.

## Notes and next steps

- Findings and the full discussion are in `docs/DISCUSSION.md`. The numeric results and figures are saved under `results/runs/imagenet1k_val/` and `results/figures/`.
- Every curve depends on the sampled images, so numbers vary slightly run to run. Raise `NUM_IMAGES` for tighter estimates and add seeds for error bars.
- The ablated (trained without PE) and RPT conditions from the guidance paper need training from scratch and are out of scope for this pretrained study.
- Next: sweep the ablation window boundary block by block to find where the decay switches on, repeat the ablation study on RoPE, add a content control (random images collapse the `mlp_zero_all` recovery that real images keep), and connect the middle block MLP features to the SAE code in `project_code/src/SAE`.